In [2]:
# Required Libraries
import os
import re
import time
import json
import random
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException

In [8]:
# =========================
# DataCenterMap scraper
# =========================


# -------------------------
# Paths
# -------------------------
BASE_DIR = Path.cwd()  # already inside data_center_map

INPUT_FILES = {
    "united_states": BASE_DIR / "03_datacentermap_united_states.csv"
}

# outputs go ONE LEVEL UP (to project root)
RAW_DATA_DIR = BASE_DIR.parent / "raw data"
CACHE_FILE = BASE_DIR.parent / "cache" / "datacentermap_scrape_cache.json"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)


# -------------------------
# Browser setup
# -------------------------
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
]

BLOCK_PATTERNS = [
    "too many requests",
    "access denied",
    "captcha",
    "you have been blocked",
    "detected unusual activity",
    "rate limit exceeded",
    "activity has been limited",
]

def make_driver():
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,2200")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument(f"--user-agent={random.choice(USER_AGENTS)}")
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(60)
    return driver


# -------------------------
# Cache helpers
# -------------------------
def load_cache(cache_file=CACHE_FILE):
    if cache_file.exists():
        with open(cache_file, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_cache(cache, cache_file=CACHE_FILE):
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


# -------------------------
# Text cleanup helpers
# -------------------------
def clean_text(value):
    if value is None:
        return None
    value = re.sub(r"\s+", " ", str(value)).strip()
    return value if value else None

def normalize_label(label):
    label = clean_text(label or "")
    label = label.lower()
    label = label.replace(":", "")
    return label

def is_blocked(html_text):
    lower_html = html_text.lower()
    return any(pattern in lower_html for pattern in BLOCK_PATTERNS)


# -------------------------
# HTML parsing helpers
# -------------------------
def extract_key_value_pairs(soup):
    """
    Collect likely label/value pairs from tables, definition lists, and generic row-like blocks.
    Returns dict[str, list[str]]
    """
    pairs = {}

    def add_pair(label, value):
        label = normalize_label(label)
        value = clean_text(value)
        if not label or not value:
            return
        pairs.setdefault(label, [])
        if value not in pairs[label]:
            pairs[label].append(value)

    # 1) Standard tables
    for table in soup.find_all("table"):
        for tr in table.find_all("tr"):
            cells = tr.find_all(["th", "td"])
            if len(cells) >= 2:
                label = cells[0].get_text(" ", strip=True)
                value = cells[1].get_text(" ", strip=True)
                add_pair(label, value)

    # 2) Definition lists
    for dl in soup.find_all("dl"):
        dts = dl.find_all("dt")
        dds = dl.find_all("dd")
        for dt, dd in zip(dts, dds):
            add_pair(dt.get_text(" ", strip=True), dd.get_text(" ", strip=True))

    # 3) Generic blocks with strong / b / span labels
    for block in soup.find_all(["div", "li", "p"]):
        text = clean_text(block.get_text(" ", strip=True))
        if not text or len(text) > 500:
            continue

        # patterns like "Address: 123 Main St"
        m = re.match(r"^([A-Za-z0-9\/\-\s\(\)&]+?)\s*:\s*(.+)$", text)
        if m:
            add_pair(m.group(1), m.group(2))
            continue

        # patterns like <strong>Address</strong> 123 Main St
        strong = block.find(["strong", "b"])
        if strong:
            label = clean_text(strong.get_text(" ", strip=True))
            full_text = clean_text(text)
            if label and full_text and full_text.lower().startswith(label.lower()):
                value = clean_text(full_text[len(label):].lstrip(": ").strip())
                if value:
                    add_pair(label, value)

    return pairs


def first_matching_value(pairs, candidate_labels):
    """
    Search extracted key/value pairs for the first matching label.
    """
    for wanted in candidate_labels:
        wanted_norm = normalize_label(wanted)
        for label, values in pairs.items():
            if wanted_norm == label or wanted_norm in label:
                if values:
                    return clean_text(values[0])
    return None


def extract_with_regex(text, patterns):
    for pattern in patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)
        if m:
            return clean_text(m.group(1))
    return None


def extract_external_links(soup, page_url):
    page_domain = urlparse(page_url).netloc.replace("www.", "").lower()
    links = []

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if not href.startswith(("http://", "https://")):
            continue

        link_domain = urlparse(href).netloc.replace("www.", "").lower()

        # Exclude self-links and common non-website targets
        if not link_domain or link_domain == page_domain:
            continue
        if "datacentermap.com" in link_domain:
            continue
        if href.startswith("mailto:") or href.startswith("tel:"):
            continue

        if href not in links:
            links.append(href)

    return links


def extract_metadata_from_html(html, page_url):
    soup = BeautifulSoup(html, "html.parser")
    page_text = clean_text(soup.get_text("\n", strip=True)) or ""

    pairs = extract_key_value_pairs(soup)

    # Address
    address = first_matching_value(
        pairs,
        [
            "address",
            "street address",
            "location",
            "postal address",
        ],
    )
    if not address:
        address = extract_with_regex(
            page_text,
            [
                r"Address\s*:\s*(.+?)(?:\n|Website\s*:|Description\s*:|White\s*space\s*:|Power\s*:|$)",
                r"Location\s*:\s*(.+?)(?:\n|Website\s*:|Description\s*:|White\s*space\s*:|Power\s*:|$)",
            ],
        )

    # Service Description
    service_description = first_matching_value(
        pairs,
        [
            "description",
            "service description",
            "services",
            "facility description",
            "overview",
        ],
    )
    if not service_description:
        service_description = extract_with_regex(
            page_text,
            [
                r"Description\s*:\s*(.+?)(?:\n[A-Z][A-Za-z\s]{1,40}\s*:|$)",
                r"Services?\s*:\s*(.+?)(?:\n[A-Z][A-Za-z\s]{1,40}\s*:|$)",
            ],
        )

    # Build-up Power Capacity
    build_up_power_capacity = first_matching_value(
        pairs,
        [
            "build-up power capacity",
            "built-up power capacity",
            "power capacity",
            "power",
            "total power",
            "it load",
            "critical load",
        ],
    )
    if not build_up_power_capacity:
        build_up_power_capacity = extract_with_regex(
            page_text,
            [
                r"Build[\-\s]*up power capacity\s*:\s*(.+?)(?:\n|$)",
                r"Built[\-\s]*up power capacity\s*:\s*(.+?)(?:\n|$)",
                r"Power capacity\s*:\s*(.+?)(?:\n|$)",
                r"Power\s*:\s*(.+?)(?:\n|$)",
            ],
        )

    # Whitespace
    whitespace = first_matching_value(
        pairs,
        [
            "white space",
            "whitespace",
            "space",
            "raised floor space",
            "technical space",
        ],
    )
    if not whitespace:
        whitespace = extract_with_regex(
            page_text,
            [
                r"White\s*space\s*:\s*(.+?)(?:\n|$)",
                r"Whitespace\s*:\s*(.+?)(?:\n|$)",
                r"Raised floor space\s*:\s*(.+?)(?:\n|$)",
            ],
        )

    # Website links
    website_links = extract_external_links(soup, page_url)

    return {
        "Address": address,
        "Service Description": service_description,
        "Build-up Power Capacity": build_up_power_capacity,
        "Whitespace": whitespace,
        "Website Links": " | ".join(website_links) if website_links else None,
    }


# -------------------------
# Selenium page fetch
# -------------------------
def fetch_page_html(driver, url, wait_seconds=20):
    driver.get(url)

    try:
        WebDriverWait(driver, wait_seconds).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
    except TimeoutException:
        pass

    # small post-load pause for dynamic content
    time.sleep(random.uniform(2.0, 4.0))

    return driver.page_source


# -------------------------
# URL scraper
# -------------------------
def scrape_datacenter_url(driver, url, cache):
    if url in cache:
        return cache[url]

    try:
        html = fetch_page_html(driver, url)

        if not html or is_blocked(html):
            result = {
                "Datacenter URL": url,
                "Address": None,
                "Service Description": None,
                "Build-up Power Capacity": None,
                "Whitespace": None,
                "Website Links": None,
                "Scrape Status": "blocked_or_empty",
            }
        else:
            metadata = extract_metadata_from_html(html, url)
            result = {
                "Datacenter URL": url,
                **metadata,
                "Scrape Status": "ok",
            }

    except TimeoutException:
        result = {
            "Datacenter URL": url,
            "Address": None,
            "Service Description": None,
            "Build-up Power Capacity": None,
            "Whitespace": None,
            "Website Links": None,
            "Scrape Status": "timeout",
        }
    except WebDriverException as e:
        result = {
            "Datacenter URL": url,
            "Address": None,
            "Service Description": None,
            "Build-up Power Capacity": None,
            "Whitespace": None,
            "Website Links": None,
            "Scrape Status": f"webdriver_error: {clean_text(e)}",
        }
    except Exception as e:
        result = {
            "Datacenter URL": url,
            "Address": None,
            "Service Description": None,
            "Build-up Power Capacity": None,
            "Whitespace": None,
            "Website Links": None,
            "Scrape Status": f"error: {clean_text(e)}",
        }

    cache[url] = result
    return result


# -------------------------
# Main regional loop
# -------------------------
cache = load_cache()
driver = make_driver()

try:
    for region_name, input_csv in INPUT_FILES.items():
        print(f"\n=== Processing: {region_name} ===")

        df_input = pd.read_csv(input_csv)

        if "Datacenter URL" not in df_input.columns:
            raise ValueError(f"'Datacenter URL' column not found in {input_csv}")

        urls = (
            df_input["Datacenter URL"]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .tolist()
        )

        scraped_rows = []

        for i, url in enumerate(urls, start=1):
            print(f"[{region_name}] {i}/{len(urls)} -> {url}")

            row = scrape_datacenter_url(driver, url, cache)
            scraped_rows.append(row)

            # save cache incrementally every 25 pages
            if i % 25 == 0:
                save_cache(cache)

            # polite delay
            time.sleep(random.uniform(3.0, 7.0))

        df_scraped = pd.DataFrame(scraped_rows)

        # merge scraped metadata back onto original regional CSV
        df_out = df_input.merge(df_scraped, on="Datacenter URL", how="left")

        output_path = RAW_DATA_DIR / f"raw_{region_name}_datacenters.csv"
        df_out.to_csv(output_path, index=False, encoding="utf-8")

        print(f"Saved {len(df_out)} rows to: {output_path}")

        # save cache after each region
        save_cache(cache)

finally:
    driver.quit()
    save_cache(cache)

print("\nDone.")


=== Processing: united_states ===
[united_states] 1/1635 -> https://www.datacentermap.com/usa/california/bakersfield/level3-bakerfield1/
[united_states] 2/1635 -> https://www.datacentermap.com/usa/california/bakersfield/nygc-bakersfield/
[united_states] 3/1635 -> https://www.datacentermap.com/usa/california/bakersfield/twtc-bakersfield/
[united_states] 4/1635 -> https://www.datacentermap.com/usa/california/burbank/3015-winona-avenue/
[united_states] 5/1635 -> https://www.datacentermap.com/usa/california/el-centro/calethos-campus-site-1/
[united_states] 6/1635 -> https://www.datacentermap.com/usa/california/el-centro/calethos-campus-site-2/
[united_states] 7/1635 -> https://www.datacentermap.com/usa/california/el-centro/calethos-site-1-building-1/
[united_states] 8/1635 -> https://www.datacentermap.com/usa/california/el-centro/calethos-site-1-building-2/
[united_states] 9/1635 -> https://www.datacentermap.com/usa/california/el-centro/calethos-site-1-building-3/
[united_states] 10/1635 